In [ ]:
#workkshopping Historical_FWI.py generalisation 

#first need to recreate the iberia work but more efficient to check data processing is working.



In [ ]:
#cylc stuff 

[task parameters]
    
    member = 1..15

[scheduling]

    [[graph]]
        P1Y = """
            hadgem_historical_fwi<member>
        """

[runtime]
    [[hadgem_historical_fwi<member>]]
        script = """
        set -eux
        conda activate sowf
        cd /data/users/bob.potts/StateOfFires_2025-26/code
        python hadgem_historical_fwi.py
        """

        platform = spice

        [[[directives]]]
            --mem = 100G
            --time = 60
            --cpus-per-task = 1
            --ntasks = 1

In [ ]:

import os
import sys
import glob
import time

import xarray as xr
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar
from xclim.indices import hot_spell_frequency


# User defined variables

UKCP_ENSEMBLE = os.environ["CYLC_TASK_PARAM_ensemble"]
CYCLE_POINT = os.environ["CYLC_TASK_CYCLE_POINT"]

# User defined variables
START_YEAR = f'{CYCLE_POINT[:4]}-01-01'
END_YEAR = f'{CYCLE_POINT[:4]}-12-31'

def main():
    print(f"ENSAMBLE = {UKCP_ENSEMBLE}")
    print(f"YEAR = {CYCLE_POINT}")
    # --------------------------------------------------------------------------
    # 1. Detect number of CPUs allocated by SLURM
    # --------------------------------------------------------------------------
    n_cpus = int(os.environ.get("SLURM_CPUS_PER_TASK", "1"))
    print(f"Detected {n_cpus} CPUs from SLURM")

    # --------------------------------------------------------------------------
    # 2. Start local Dask cluster for parallel processing
    # --------------------------------------------------------------------------
    cluster = LocalCluster(
        n_workers=n_cpus,
        threads_per_worker=1,  # ideal for xarray/numpy operations
        memory_limit="auto",
        processes=True,         # ensures true parallelism
    )
    client = Client(cluster)
    print("Cluster dashboard:", client.dashboard_link)

    # --------------------------------------------------------------------------
    # 3. Open multi-year daily temperature data
    # --------------------------------------------------------------------------
    ensemble_str = f"{int(UKCP_ENSEMBLE):02d}"
    input_files = sorted(glob.glob(f"/data/users/ukcpsis/ukcp/land-cpm/uk/2.2km/rcp85/{ensemble_str}/tas/1hr/v20210615/*.nc"))

    # Select files by date range (defined above )
    selected_files = UKCP_date_filter(input_files, START_YEAR, END_YEAR)
    print(f"Selected {len(selected_files)} files for date range {START_YEAR} to {END_YEAR}")

    tas_dataset = xr.open_mfdataset(
        selected_files,
        combine="by_coords",
        chunks={"time": 100, "grid_latitude": 100, "grid_longitude": 100},
    )["tas"]  # extract the variable

    # --------------------------------------------------------------------------
    # 4. Load threshold dataset
    # --------------------------------------------------------------------------
    threshold_ds = xr.open_dataset("/data/users/urbanclima/MACC/thresholds/thresholds_UKCP_dmax_tas_ensamble01_1980_2000.nc")
    threshold_var = "quantile"  # change variable if needed - Currently Unused

    # pick the first data variable if you don't know its name
    data_var = list(threshold_ds.data_vars)[0]
    print("Using data variable:", data_var)

    thresh = threshold_ds[data_var]   # this is the DataArray of thresholds (with dim 'quantile')
    print(thresh)

    if "quantile" in thresh.dims:
        # ensure quantiles are in ascending order
        thresh = thresh.sortby("quantile")
        threshold_90 = thresh.sel(quantile=0.9, method="nearest")
        threshold_10 = thresh.sel(quantile=0.1, method="nearest")
    else:
        # no quantile dim: threshold already single-valued
        threshold_90 = thresh
        threshold_10 = thresh

    print("threshold_90 mean:", threshold_90.mean().item())
    print("threshold_10 mean:", threshold_10.mean().item())
    print("threshold_90 min/max:", float(threshold_90.min()), float(threshold_90.max()))

    # Drop the quantile coordinate if it exists
    if "quantile" in threshold_90.coords:
        threshold_90 = threshold_90.drop_vars("quantile")

    # Ensure units are consistent
    tas_dataset.attrs["units"] = "degC"
    threshold_90.attrs["units"] = "degC"

    # --------------------------------------------------------------------------
    # 5. Compute daily maximum temperature and align with threshold
    # --------------------------------------------------------------------------
    tas_daily_max = tas_dataset.resample(time="1D").max()
    tas_daily_max, threshold_90 = xr.align(tas_daily_max, threshold_90)

    # --------------------------------------------------------------------------
    # 6. Calculate monthly hot spell frequency
    # --------------------------------------------------------------------------
    hot_spell_freq = hot_spell_frequency(
        tasmax=tas_daily_max,
        thresh=threshold_90,
        window=3,  # minimum consecutive hot days
        freq="M",
    )

    # Compute the result with progress bar
    with ProgressBar():
        hot_spell_freq_computed = hot_spell_freq.compute()

    # Assign proper variable name and save to NetCDF
    hot_spell_freq_computed.name = "hot_spell_frequency"
    hot_spell_freq_computed.to_netcdf(
        f"/data/users/urbanclima/MACC/indices/hot_spell_frequency/ukcp/hot_spell_frequency_tas_{ensemble_str}_90pc_3days_{CYCLE_POINT[:4]}.nc"
        )


# ==============================================================================
# Entry point
# ==============================================================================
if __name__ == "__main__":
    start_time = time.time()
    main()
    print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
#CONFIG

COUNTRY = 'Iberia'
# Options: 'South Korea' (3), 'Iberia' (8), 'Scotland' (7)
############# User inputs end here #############


folder = '/data/scratch/chantelle.burton/SoW2526/'

#Set up the 2025 files and months automatically
if COUNTRY == 'South Korea':
    print('Running South Korea')
    Month = 3
    month = 'March'
    percentile = 95
    daterange = iris.Constraint(time=lambda cell: cell.point.month == Month)
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-01-01-2025-05-31_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')
      

elif COUNTRY == 'Iberia':
    print('Running Iberia')
    Month = 8
    month = 'Aug'
    percentile = 95
    daterange = iris.Constraint(time=lambda cell: cell.point.month == Month)
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-06-01-2025-10-01_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')

elif COUNTRY == 'Scotland':
    print('Running Scotland')
    Month = 7
    month = 'July'
    percentile = 95
    daterange = iris.Constraint(time=lambda cell: cell.point.month == Month)
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-06-01-2025-10-01_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')




In [ ]:
# Create .dat files for unbias-corrected historical data, then plot  PDFs

#module load scitools/default-current
#python3
#-*- coding: iso-8859-1 -*-


import numpy as np
import iris
import time
#matplotlib.use('Agg')
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
import os
import sys
from utils.constrain_cubes_standard import *
from utils.cubefuncs import *


############# User inputs here #############
Country = 'Iberia'
# Options: 'South Korea' (3), 'Iberia' (8), 'Scotland' (7)
############# User inputs end here #############


folder = '/data/scratch/chantelle.burton/SoW2526/'

if Country == 'Iberia':
    print('Running Iberia')
    Month = 8
    month = 'Aug'
    percentile = 95
    daterange = iris.Constraint(time=lambda cell: cell.point.month == Month)
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-06-01-2025-10-01_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')

#member = os.environ["CYLC_TASK_PARAM_member"]
#member = 10
for member in np.arange(10,11):
    HadGEM3_Arr = []
    start_time = time.time()
    for year in np.arange(1960, 2014):
        print('Member number:',member,'\n Year:',year)
        HadGEM3 = iris.load_cube(folder+'/historicalFWI/HadGEM/FWI_HadGEM3-A-N216_r1i1p'+str(member)+'_historical_gwl'+str(year)+'0'+str(Month)+'01*.nc', 'canadian_fire_weather_index')
        #HadGEM3 = iris.load_cube(folder+'/historicalFWI/HadGEM/FWI_HadGEM3-A-N216_r1i1p10_historical_gwl'+str(year)+'0'+str(Month)+'01*.nc', 'canadian_fire_weather_index')#TEST
        if HadGEM3 == None:
            pass
        else:
            daterange = iris.Constraint(time=lambda cell: cell.point.year == year)
        if HadGEM3 == None:
            pass
        else:
            HadGEM3 = HadGEM3.extract(daterange)
        if HadGEM3 == None:
            pass
        else:            
            HadGEM3 = HadGEM3.extract(daterange)
        if HadGEM3 == None:
            pass
        else:
            tcoord = HadGEM3.coord('time')
            dates = tcoord.units.num2date(tcoord.points)
            print("Selected times:\n", dates[0], "to", dates[-1])
            HadGEM3 = TimePercentile(HadGEM3, percentile)
            HadGEM3 = contrain_to_sow_shapefile(HadGEM3, '/data/users/chantelle.burton/Attribution/StateOfFires_2025-26/SoW2526_Focal_MASTER_20260218.shp','Northwest Iberia')
            #HadGEM3 = CountryConstrain(HadGEM3, Country)
            HadGEM3 = CountryPercentile(HadGEM3, percentile)
            HadGEM3 = np.ravel(HadGEM3.data)
            HadGEM3_Arr.append(HadGEM3)


    #Save HaGEM3 text out to a file
    f = open('/data/scratch/bob.potts/sowf/test_output/HadGEM3_FWI_1960-2013_'+Country+'_'+str(member)+'_'+str(percentile)+'%.dat','a')
    np.savetxt(f,(HadGEM3_Arr))
    f.close()  
    print('Finished')
#exit()

print("--- %s seconds ---" % (np.round(time.time() - start_time, 2)))
#single member takes approx 8 minutes.

Running Iberia
Member number: 10 
 Year: 1960
Selected times:
 1960-08-01 12:00:00 to 1960-08-30 12:00:00
Member number: 10 
 Year: 1961
Selected times:
 1961-08-01 12:00:00 to 1961-08-30 12:00:00
Member number: 10 
 Year: 1962
Selected times:
 1962-08-01 12:00:00 to 1962-08-30 12:00:00
Member number: 10 
 Year: 1963
Selected times:
 1963-08-01 12:00:00 to 1963-08-30 12:00:00
Member number: 10 
 Year: 1964
Selected times:
 1964-08-01 12:00:00 to 1964-08-30 12:00:00
Member number: 10 
 Year: 1965
Selected times:
 1965-08-01 12:00:00 to 1965-08-30 12:00:00
Member number: 10 
 Year: 1966
Selected times:
 1966-08-01 12:00:00 to 1966-08-30 12:00:00
Member number: 10 
 Year: 1967
Selected times:
 1967-08-01 12:00:00 to 1967-08-30 12:00:00
Member number: 10 
 Year: 1968
Selected times:
 1968-08-01 12:00:00 to 1968-08-30 12:00:00
Member number: 10 
 Year: 1969
Selected times:
 1969-08-01 12:00:00 to 1969-08-30 12:00:00
Member number: 10 
 Year: 1970
Selected times:
 1970-08-01 12:00:00 to 1970